In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tlo import Date
from tlo.analysis.utils import (
    extract_results,
    summarize,
)
import matplotlib.dates as mdates

In [15]:
results_folder = Path(
    '/Users/rem76/PycharmProjects/TLOmodel/outputs/rm916@ic.ac.uk/climate_scenario_runs_lhs_param_scan-2026-02-03T101537Z')

output_folder = Path(
    '/Users/rem76/PycharmProjects/TLOmodel/outputs/rm916@ic.ac.uk/climate_scenario_runs_lhs_param_scan-2026-02-03T101537Z')


In [16]:
climate_sensitivity_analysis = False
main_text = False
parameter_uncertainty_analysis = True

if parameter_uncertainty_analysis:
    scenario_names = range(0, 200, 1)
    scenarios_of_interest = scenario_names
    suffix = "parameter_UA"
if main_text:
    scenario_names = [
        "Baseline",
        "Best Case",
        "Worst Case"
    ]
    suffix = "main_text"
    scenarios_of_interest = [0, 1]  #, 2]

In [18]:

min_year = 2025
max_year = 2028
spacing_of_years = 1

In [19]:
num_cancelled_by_facility[9]

NameError: name 'num_cancelled_by_facility' is not defined

In [ ]:
target_year_sequence = range(min_year, max_year, spacing_of_years)


# Define the extraction functions (add these near your other helper functions)
def get_num_treatments_total_delayed(_df):
    """Count total number of delayed HSI events"""
    _df["date"] = pd.to_datetime(_df["date"])
    _df = _df.loc[_df["date"].between(*TARGET_PERIOD)]
    return pd.Series(len(_df), name="total")


def get_num_treatments_total_cancelled(_df):
    """Count total number of cancelled HSI events"""
    _df["date"] = pd.to_datetime(_df["date"])
    _df = _df.loc[_df["date"].between(*TARGET_PERIOD)]
    return pd.Series(len(_df), name="total")


def get_num_treatments_total(_df):
    """Sum all treatment counts"""
    _df["date"] = pd.to_datetime(_df["date"])
    _df = _df.loc[_df["date"].between(*TARGET_PERIOD)]
    total = {}
    for d in _df["hsi_event_key_to_counts"]:
        for k, v in d.items():
            total[k] = total.get(k, 0) + v
    print(sum(total.values()))
    return pd.Series(sum(total.values()), name="total")


def get_num_dalys_total(_df):
    """Return total number of DALYS (Stacked) by label (total by age-group within the TARGET_PERIOD)
    """
    return pd.Series(_df \
                     .loc[_df.year.between(*[i.year for i in TARGET_PERIOD])] \
                     .drop(columns=['date', 'sex', 'age_range', 'year']) \
                     .sum().sum(), name="total")


def get_num_dalys_by_month(_df):
    """Sum all DALYs across all causes by month for the target year"""
    _df["date"] = pd.to_datetime(_df["date"])
    _df = _df.loc[_df["date"].between(*TARGET_PERIOD)]

    # Sum across all disease columns (excluding non-disease columns)
    disease_columns = [col for col in _df.columns
                       if col not in ['age_range', 'month', 'sex', 'year', 'date']]

    # Group by month and sum
    monthly_dalys = _df.groupby('month')[disease_columns].sum().sum(axis=1)

    return monthly_dalys


def get_population_for_year(_df):
    """Returns the population in the year of interest"""
    _df["date"] = pd.to_datetime(_df["date"])

    # Filter the DataFrame based on the target period
    filtered_df = _df.loc[_df["date"].between(*TARGET_PERIOD)]
    numeric_df = filtered_df.drop(columns=["female", "male"], errors="ignore")
    population_sum = numeric_df.sum(numeric_only=True)

    return population_sum


results_folder_baseline = Path(
    '/Users/rem76/PycharmProjects/TLOmodel/outputs/rm916@ic.ac.uk/baseline_run_with_pop-2026-02-05T113912Z')
results_folder_climate = Path(
    '/Users/rem76/PycharmProjects/TLOmodel/outputs/rm916@ic.ac.uk/climate_scenario_runs_lhs_param_scan-2026-02-03T101537Z')


# Helper functions
def get_hsi_counts_by_facility_monthly(_df):
    """Get HSI counts by facility_id and month"""
    _df["date"] = pd.to_datetime(_df["date"])
    _df = _df.loc[_df["date"].between(*TARGET_PERIOD)]

    if len(_df) == 0:
        return pd.Series(dtype=int)

    # Create keys like "2010-02_128" (month_facilityID)
    facility_month_totals = {}
    for _, row in _df.iterrows():
        month_key = row['date'].strftime('%Y-%m')
        counts_dict = row['counts']

        for key, value in counts_dict.items():
            if ':' in key:
                facility_id, _ = key.split(':', 1)
                facility_month_totals[facility_id] = facility_month_totals.get(facility_id, 0) + value

    return pd.Series(facility_month_totals)


def get_num_treatments_by_facility_delayed(_df):
    """Count number of delayed HSI events by facility"""
    print(_df)
    _df["date"] = pd.to_datetime(_df["date"])
    _df = _df.loc[_df["date"].between(*TARGET_PERIOD)]

    if len(_df) == 0:
        return pd.Series(dtype=int, name="delayed_count")

    # Group by Facility_ID and count
    delayed_by_facility = _df.groupby('Facility_ID').size()
    delayed_by_facility.name = 'delayed_count'

    return delayed_by_facility


def get_num_treatments_by_facility_cancelled(_df):
    """Count number of cancelled HSI events by facility"""
    _df["date"] = pd.to_datetime(_df["date"])
    _df = _df.loc[_df["date"].between(*TARGET_PERIOD)]

    if len(_df) == 0:
        return pd.Series(dtype=int, name="cancelled_count")

    # Group by Facility_ID and count
    cancelled_by_facility = _df.groupby('Facility_ID').size()
    cancelled_by_facility.name = 'cancelled_count'

    return cancelled_by_facility


# Storage dictionaries
all_scenarios_appointment_delayed_mean = {}
all_scenarios_appointment_cancelled_mean = {}
all_scenarios_dalys_mean = {}
# Get denominators
baseline_hsi_by_facility = {}

for target_year in target_year_sequence:
    TARGET_PERIOD = (Date(target_year, 1, 1), Date(target_year, 12, 31))

    # Get baseline HSI by facility from BASELINE folder (DENOMINATOR)
    hsi_by_facility = summarize(extract_results(
        results_folder_baseline,  # ← Still from baseline
        module='tlo.methods.healthsystem.summary',
        key='hsi_event_counts_by_facility_monthly',
        custom_generate_series=get_hsi_counts_by_facility_monthly,
        do_scaling=False
    ), only_mean=True, collapse_columns=False)[0]

    # Fix index to be int
    hsi_by_facility.index = pd.Index([int(x) for x in hsi_by_facility.index], name='Facility_ID')
    baseline_hsi_by_facility[target_year] = hsi_by_facility

# Main loop
for draw in range(len(scenarios_of_interest)):
    print(draw)
    all_years_data_delayed_mean = {}
    all_years_data_cancelled_mean = {}
    all_years_dalys_mean = {}

    for target_year in target_year_sequence:
        TARGET_PERIOD = (Date(target_year, 1, 1), Date(target_year, 12, 31))

        # Get delayed counts BY FACILITY from CLIMATE folder (NUMERATOR)
        num_delayed_by_facility = summarize(extract_results(
            results_folder_climate,  # ← Changed from results_folder_baseline
            module='tlo.methods.healthsystem.summary',
            key='Weather_delayed_HSI_Event_full_info',
            custom_generate_series=get_num_treatments_by_facility_delayed,
            do_scaling=False
        ), only_mean=True, collapse_columns=False)  #[draw]

        # Get cancelled counts BY FACILITY from CLIMATE folder (NUMERATOR)
        num_cancelled_by_facility = summarize(extract_results(
            results_folder_climate,  # ← Changed from results_folder_baseline
            module='tlo.methods.healthsystem.summary',
            key='Weather_cancelled_HSI_Event_full_info',
            custom_generate_series=get_num_treatments_by_facility_cancelled,
            do_scaling=False
        ), only_mean=True, collapse_columns=False)  #[draw]

        # Align and divide by baseline (from baseline folder)
        baseline_aligned, delayed_aligned = baseline_hsi_by_facility[target_year].align(num_delayed_by_facility,
                                                                                        fill_value=0)
        delayed_proportions = delayed_aligned / baseline_aligned

        baseline_aligned, cancelled_aligned = baseline_hsi_by_facility[target_year].align(num_cancelled_by_facility,
                                                                                          fill_value=0)
        cancelled_proportions = cancelled_aligned / baseline_aligned

        all_years_data_delayed_mean[target_year] = delayed_proportions
        all_years_data_cancelled_mean[target_year] = cancelled_proportions

    all_scenarios_appointment_delayed_mean[draw] = all_years_data_delayed_mean
    all_scenarios_appointment_cancelled_mean[draw] = all_years_data_cancelled_mean
    all_scenarios_dalys_mean[draw] = all_years_dalys_mean

In [ ]:
# %% [markdown]
# # PRCC Analysis for Climate Disruption Sensitivity Analysis
#
# Adapted for results indexed by draw number with facility-level proportions.

# %% Cell 1: Imports and Configuration
import json
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings

# Configuration - update these paths
LHS_FILE = "/Users/rem76/PycharmProjects/TLOmodel/src/scripts/climate_disruptions/lhs_parameter_draws.json"
n_samples_to_use = 200
start_index = 0

# Parameters that were varied in LHS
LHS_PARAMS = [
    "scale_factor_prob_disruption",
    "delay_in_seeking_care_weather",
    "scale_factor_reseeking_healthcare_post_disruption",
    "scale_factor_appointment_urgency",
    "scale_factor_severity_disruption_and_delay",
]

# %% Cell 2: Load LHS Parameters
with open(LHS_FILE, 'r') as f:
    LHS_grid_full = json.load(f)

parameter_grid = LHS_grid_full[start_index:start_index + n_samples_to_use]
lhs_params = pd.DataFrame(parameter_grid)[LHS_PARAMS]

print(f"Loaded {len(lhs_params)} LHS parameter draws")
print(f"Parameters: {lhs_params.columns.tolist()}")
print(lhs_params.head())


# %% Cell 3: Core PRCC Function
def partial_rank_correlation(X: np.ndarray, y: np.ndarray) -> tuple:
    """
    Calculate Partial Rank Correlation Coefficients (PRCC) for each input parameter.
    """
    n_samples, n_params = X.shape

    X_ranked = np.apply_along_axis(stats.rankdata, 0, X)
    y_ranked = stats.rankdata(y)

    prcc_values = np.zeros(n_params)
    p_values = np.zeros(n_params)

    for i in range(n_params):
        xi = X_ranked[:, i]
        other_indices = [j for j in range(n_params) if j != i]
        X_others = X_ranked[:, other_indices]

        if len(other_indices) > 0:
            X_others_with_intercept = np.column_stack([np.ones(n_samples), X_others])
            coeffs_xi = np.linalg.lstsq(X_others_with_intercept, xi, rcond=None)[0]
            xi_residuals = xi - X_others_with_intercept @ coeffs_xi
            coeffs_y = np.linalg.lstsq(X_others_with_intercept, y_ranked, rcond=None)[0]
            y_residuals = y_ranked - X_others_with_intercept @ coeffs_y
        else:
            xi_residuals = xi
            y_residuals = y_ranked

        correlation, p_value = stats.pearsonr(xi_residuals, y_residuals)
        prcc_values[i] = correlation
        p_values[i] = p_value

    return prcc_values, p_values


# %% Cell 4: Aggregate Facility-Level Data to Single Value Per Draw
def aggregate_facility_data(data_by_draw, target_years, agg_func='mean'):
    """
    Convert facility-level proportions to a single value per draw per year.

    Parameters:
    -----------
    data_by_draw : dict
        Dictionary: draw -> year -> Series/DataFrame of facility proportions
    target_years : list
        List of years
    agg_func : str
        Aggregation function: 'mean', 'median', 'sum', 'max'

    Returns:
    --------
    dict : draw -> year -> single aggregated value
    """
    aggregated = {}

    for draw, year_data in data_by_draw.items():
        aggregated[draw] = {}
        for year, values in year_data.items():
            if isinstance(values, (pd.Series, pd.DataFrame)):
                # Remove inf and nan before aggregating
                clean_values = values.replace([np.inf, -np.inf], np.nan).dropna()
                if len(clean_values) > 0:
                    if agg_func == 'mean':
                        aggregated[draw][year] = clean_values.mean()
                    elif agg_func == 'median':
                        aggregated[draw][year] = clean_values.median()
                    elif agg_func == 'sum':
                        aggregated[draw][year] = clean_values.sum()
                    elif agg_func == 'max':
                        aggregated[draw][year] = clean_values.max()
                    else:
                        aggregated[draw][year] = clean_values.mean()
                else:
                    aggregated[draw][year] = np.nan
            elif isinstance(values, (int, float)):
                aggregated[draw][year] = values
            else:
                aggregated[draw][year] = np.nan

    return aggregated


# %% Cell 5: PRCC Calculation Functions
def calculate_prcc_by_year(lhs_params, output_by_draw, target_years):
    """Calculate PRCC for each year separately."""
    X = lhs_params.values
    param_names = lhs_params.columns.tolist()
    n_draws = len(lhs_params)
    results = []

    for year in target_years:
        y = np.full(n_draws, np.nan)

        for draw_idx in range(n_draws):
            if draw_idx in output_by_draw and year in output_by_draw[draw_idx]:
                val = output_by_draw[draw_idx][year]
                if isinstance(val, (int, float)) and not np.isnan(val) and not np.isinf(val):
                    y[draw_idx] = val

        valid_mask = ~np.isnan(y)
        n_valid = valid_mask.sum()

        if n_valid < 10:
            warnings.warn(f"Year {year}: Only {n_valid} valid draws, skipping")
            continue

        prcc, pvals = partial_rank_correlation(X[valid_mask], y[valid_mask])

        for i, param in enumerate(param_names):
            results.append({
                'year': year,
                'parameter': param,
                'prcc': prcc[i],
                'p_value': pvals[i],
                'significant': pvals[i] < 0.05,
                'n_valid': n_valid
            })

    return pd.DataFrame(results)


def calculate_prcc_pooled(lhs_params, output_by_draw, target_years):
    """Calculate PRCC pooled across all years."""
    param_names = lhs_params.columns.tolist()
    n_draws = len(lhs_params)

    X_pooled, y_pooled = [], []

    for draw_idx in range(n_draws):
        if draw_idx not in output_by_draw:
            continue
        for year in target_years:
            if year in output_by_draw[draw_idx]:
                val = output_by_draw[draw_idx][year]
                if isinstance(val, (int, float)) and not np.isnan(val) and not np.isinf(val):
                    X_pooled.append(lhs_params.iloc[draw_idx].values)
                    y_pooled.append(val)

    if len(y_pooled) < 10:
        warnings.warn(f"Only {len(y_pooled)} valid observations, insufficient for PRCC")
        return pd.DataFrame()

    X_pooled = np.array(X_pooled)
    y_pooled = np.array(y_pooled)

    prcc, pvals = partial_rank_correlation(X_pooled, y_pooled)

    results = []
    for i, param in enumerate(param_names):
        results.append({
            'year': 'all_years',
            'parameter': param,
            'prcc': prcc[i],
            'p_value': pvals[i],
            'significant': pvals[i] < 0.05,
            'n_valid': len(y_pooled)
        })

    return pd.DataFrame(results)


def calculate_prcc(lhs_params, output_by_draw, target_years, include_pooled=True):
    """Calculate PRCC by year and optionally pooled across all years."""
    yearly = calculate_prcc_by_year(lhs_params, output_by_draw, target_years)

    if include_pooled:
        pooled = calculate_prcc_pooled(lhs_params, output_by_draw, target_years)
        if len(pooled) > 0:
            return pd.concat([yearly, pooled], ignore_index=True)

    return yearly


# %% Cell 6: Run PRCC Analysis
# Aggregate facility-level data to single values per draw
# Choose aggregation method: 'mean', 'median', 'sum', 'max'
AGG_METHOD = 'mean'

target_years = list(target_year_sequence)

# Aggregate the facility-level proportions
delayed_aggregated = aggregate_facility_data(
    all_scenarios_appointment_delayed_mean,
    target_years,
    agg_func=AGG_METHOD
)

cancelled_aggregated = aggregate_facility_data(
    all_scenarios_appointment_cancelled_mean,
    target_years,
    agg_func=AGG_METHOD
)

# For DALYs (if populated)
dalys_aggregated = aggregate_facility_data(
    all_scenarios_dalys_mean,
    target_years,
    agg_func=AGG_METHOD
)

print(f"Aggregated data using '{AGG_METHOD}' method")
print(f"Number of draws with delayed data: {len(delayed_aggregated)}")
print(f"Number of draws with cancelled data: {len(cancelled_aggregated)}")

# %% Cell 7: Calculate PRCC
prcc_delayed = calculate_prcc(lhs_params, delayed_aggregated, target_years)
prcc_cancelled = calculate_prcc(lhs_params, cancelled_aggregated, target_years)

# Only calculate DALYs PRCC if data exists
if any(len(v) > 0 for v in dalys_aggregated.values()):
    prcc_dalys = calculate_prcc(lhs_params, dalys_aggregated, target_years)
else:
    prcc_dalys = pd.DataFrame()
    print("No DALYs data available for PRCC")


# %% Cell 8: Display Results
def print_prcc_summary(prcc_df, output_name):
    """Print formatted PRCC summary."""
    if len(prcc_df) == 0:
        print(f"\n{output_name}: No results")
        return

    print(f"\n{'=' * 60}")
    print(f"PRCC Results for {output_name}")
    print('=' * 60)

    yearly_df = prcc_df[prcc_df['year'] != 'all_years']
    pooled_df = prcc_df[prcc_df['year'] == 'all_years']

    if len(yearly_df) > 0:
        pivot = yearly_df.pivot_table(index='parameter', columns='year', values='prcc')
        print("\nPRCC by Year:")
        print(pivot.round(3).to_string())

    if len(pooled_df) > 0:
        print("\nPRCC Pooled Across All Years:")
        for _, row in pooled_df.sort_values('prcc', key=abs, ascending=False).iterrows():
            sig = "*" if row['significant'] else ""
            print(f"  {row['parameter']}: {row['prcc']:.3f}{sig} (n={row['n_valid']})")


print_prcc_summary(prcc_delayed, "Delayed Appointments (proportion)")
print_prcc_summary(prcc_cancelled, "Cancelled Appointments (proportion)")
if len(prcc_dalys) > 0:
    print_prcc_summary(prcc_dalys, "DALYs")


# %% Cell 9: Tornado Plot Visualization
def plot_prcc_tornado(prcc_dict, figsize=(12, 5)):
    """Create tornado plots for PRCC results."""
    # Filter out empty dataframes
    prcc_dict = {k: v for k, v in prcc_dict.items() if len(v) > 0}
    n_plots = len(prcc_dict)

    if n_plots == 0:
        print("No data to plot")
        return None

    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    for ax, (name, df) in zip(axes, prcc_dict.items()):
        pooled = df[df['year'] == 'all_years']
        if len(pooled) > 0:
            prcc_vals = pooled.set_index('parameter')['prcc'].sort_values()
            subtitle = "(Pooled)"
        else:
            prcc_vals = df.groupby('parameter')['prcc'].mean().sort_values()
            subtitle = "(Mean)"

        colors = ['#d62728' if x < 0 else '#1f77b4' for x in prcc_vals.values]
        bars = ax.barh(range(len(prcc_vals)), prcc_vals.values, color=colors, alpha=0.8)

        ax.set_yticks(range(len(prcc_vals)))
        ax.set_yticklabels([p.replace('scale_factor_', '').replace('_', '\n')
                            for p in prcc_vals.index], fontsize=9)
        ax.axvline(0, color='black', linewidth=0.5)
        ax.axvline(0.3, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
        ax.axvline(-0.3, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
        ax.set_xlabel('PRCC')
        ax.set_title(f'{name}\n{subtitle}')
        ax.set_xlim(-1, 1)

    plt.tight_layout()
    return fig


fig = plot_prcc_tornado({
    'Delayed': prcc_delayed,
    'Cancelled': prcc_cancelled,
    'DALYs': prcc_dalys
})


# plt.savefig('prcc_tornado.png', dpi=300, bbox_inches='tight')

# %% Cell 10: Heatmap by Year
def plot_prcc_heatmap(prcc_dict, figsize=(12, 5)):
    """Create heatmaps showing PRCC by year."""
    prcc_dict = {k: v for k, v in prcc_dict.items() if len(v) > 0}
    n_plots = len(prcc_dict)

    if n_plots == 0:
        print("No data to plot")
        return None

    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    for ax, (name, df) in zip(axes, prcc_dict.items()):
        yearly = df[df['year'] != 'all_years']
        if len(yearly) == 0:
            ax.set_title(f'{name}\n(No yearly data)')
            continue

        pivot = yearly.pivot_table(index='parameter', columns='year', values='prcc')

        im = ax.imshow(pivot.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([p.replace('scale_factor_', '').replace('_', '\n')
                            for p in pivot.index], fontsize=9)
        ax.set_xlabel('Year')
        ax.set_title(f'{name}')

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                color = 'white' if abs(val) > 0.5 else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        color=color, fontsize=8)

    plt.colorbar(im, ax=axes, shrink=0.6, label='PRCC')
    plt.tight_layout()
    return fig


fig = plot_prcc_heatmap({
    'Delayed': prcc_delayed,
    'Cancelled': prcc_cancelled,
    'DALYs': prcc_dalys
})


# plt.savefig('prcc_heatmap.png', dpi=300, bbox_inches='tight')

# %% Cell 11: Summary Table
def create_summary_table(prcc_dict):
    """Create a summary table of pooled PRCC values."""
    rows = []
    for output_name, df in prcc_dict.items():
        if len(df) == 0:
            continue
        pooled = df[df['year'] == 'all_years']
        if len(pooled) > 0:
            for _, row in pooled.iterrows():
                rows.append({
                    'Output': output_name,
                    'Parameter': row['parameter'],
                    'PRCC': row['prcc'],
                    'p-value': row['p_value'],
                    'Significant': '***' if row['p_value'] < 0.001 else
                    '**' if row['p_value'] < 0.01 else
                    '*' if row['p_value'] < 0.05 else ''
                })

    if not rows:
        return pd.DataFrame()

    summary = pd.DataFrame(rows)
    pivot = summary.pivot_table(index='Parameter', columns='Output',
                                values='PRCC', aggfunc='first').round(3)
    return pivot


summary_table = create_summary_table({
    'Delayed': prcc_delayed,
    'Cancelled': prcc_cancelled,
    'DALYs': prcc_dalys
})
print("\nSummary Table (Pooled PRCC):")
print(summary_table)


# %% Cell 12: Export Results
def save_results(prcc_dict, prefix='prcc_results'):
    """Save all PRCC results to CSV."""
    for name, df in prcc_dict.items():
        if len(df) > 0:
            filename = f"{prefix}_{name.lower()}.csv"
            df.to_csv(filename, index=False)
            print(f"Saved {filename}")


# Uncomment to save:
# save_results({
#     'delayed': prcc_delayed,
#     'cancelled': prcc_cancelled,
#     'dalys': prcc_dalys
# })

# %% Cell 13: Optional - Check Data Quality
def check_data_quality(data_by_draw, target_years, name="Data"):
    """Check for missing/invalid values in the aggregated data."""
    n_draws = len(data_by_draw)
    n_years = len(target_years)

    valid_count = 0
    nan_count = 0
    inf_count = 0
    missing_count = 0

    for draw in range(n_draws):
        if draw not in data_by_draw:
            missing_count += n_years
            continue
        for year in target_years:
            if year not in data_by_draw[draw]:
                missing_count += 1
            else:
                val = data_by_draw[draw][year]
                if np.isnan(val):
                    nan_count += 1
                elif np.isinf(val):
                    inf_count += 1
                else:
                    valid_count += 1

    total = n_draws * n_years
    print(f"\n{name} Quality Check:")
    print(f"  Total expected: {total}")
    print(f"  Valid: {valid_count} ({100 * valid_count / total:.1f}%)")
    print(f"  NaN: {nan_count} ({100 * nan_count / total:.1f}%)")
    print(f"  Inf: {inf_count} ({100 * inf_count / total:.1f}%)")
    print(f"  Missing: {missing_count} ({100 * missing_count / total:.1f}%)")


check_data_quality(delayed_aggregated, target_years, "Delayed")
check_data_quality(cancelled_aggregated, target_years, "Cancelled")

In [24]:
# ------------------------------------------------------------------
# LOAD PREDICTED DISRUPTION RESOURCE FILE
# ------------------------------------------------------------------

df_dis = pd.read_csv(
    "/Users/rem76/PycharmProjects/TLOmodel/resources/climate_change_impacts"
    "/ResourceFile_Precipitation_Disruptions_ssp245_lowest.csv",
)

print("\nDisruption resource file shape:", df_dis.shape)
print(df_dis.head(3))

# ------------------------------------------------------------------
# FILTER TO STUDY PERIOD AND WET SEASON
# ------------------------------------------------------------------

df_dis = df_dis[
    (df_dis["year"] >= min_year)
    & (df_dis["year"] <= max_year - 1)
    & (df_dis["month"].isin([11, 12, 1, 2, 3, 4]))
    ]

# Optionally subset to tlo_facilities
if tlo_facilities is not None:
    df_dis = df_dis[df_dis["RealFacility_ID"].isin(tlo_facilities)]

# ------------------------------------------------------------------
# FLAG HIGH-PRECIP MONTHS
# ------------------------------------------------------------------

df_dis["ym"] = (
    df_dis["year"].astype(str)
    + "-"
    + df_dis["month"].astype(str).str.zfill(2)
)

df_dis["in_high_precip"] = df_dis["ym"].isin(high_rain_months)

print(f"\nRows flagged as high-precip: {df_dis['in_high_precip'].sum()}")
print(f"Rows flagged as normal:      {(~df_dis['in_high_precip']).sum()}")

# ------------------------------------------------------------------
# COMPARE PREDICTED DISRUPTION: HIGH VS NORMAL MONTHS
# ------------------------------------------------------------------

high_mean = df_dis[df_dis["in_high_precip"]]["mean_all_service"].mean()
normal_mean = df_dis[~df_dis["in_high_precip"]]["mean_all_service"].mean()

print(f"\nPredicted disruption probability (resource file):")
print(f"  High-precip months:  {high_mean:.4f}  ({high_mean * 100:.2f}%)")
print(f"  Normal wet months:   {normal_mean:.4f}  ({normal_mean * 100:.2f}%)")
print(f"  Ratio (high/normal): {high_mean / normal_mean:.3f}")

# Per-month breakdown
print("\nMean predicted disruption by YYYY-MM (high-precip months only):")
print(
    df_dis[df_dis["in_high_precip"]]
    .groupby("ym")["mean_all_service"]
    .mean()
    .sort_index()
    .to_string()
)

# Distribution comparison
print("\nPredicted disruption distribution — high-precip months:")
print(df_dis[df_dis["in_high_precip"]]["mean_all_service"].describe())
print("\nPredicted disruption distribution — normal wet months:")
print(df_dis[~df_dis["in_high_precip"]]["mean_all_service"].describe())


Disruption resource file shape: (209472, 7)
   Unnamed: 0               RealFacility_ID  year  month service  disruption  \
0           0             Mary Mount Clinic  2025      1     all    0.020390   
1           1  Edingeni Health Centre (COM)  2025      1     all    0.000000   
2           2                      KAVUKUKU  2025      1     all    0.009245   

   mean_all_service  
0          0.020390  
1          0.000000  
2          0.009245  

Rows flagged as high-precip: 10910
Rows flagged as normal:      93826

Predicted disruption probability (resource file):
  High-precip months:  0.0068  (0.68%)
  Normal wet months:   0.0094  (0.94%)
  Ratio (high/normal): 0.717

Mean predicted disruption by YYYY-MM (high-precip months only):
ym
2025-01    0.001726
2028-01    0.000517
2028-02    0.003195
2029-01    0.012695
2031-01    0.001708
2033-01    0.006744
2035-01    0.008009
2037-01    0.020835
2038-01    0.002844
2039-01    0.009259

Predicted disruption distribution — high-precip 

In [20]:
# ------------------------------------------------------------------
# SEE THE NATIONAL AVERAGE PRECIPITATION FOR A SPECIFIC MONTH
# ------------------------------------------------------------------

selected_month = "2031-12"

if selected_month not in national_mean.index:
    print(f"{selected_month} not found")
else:

avg_precip = national_mean.loc[selected_month]

print(f"National mean precipitation in {selected_month}:")
print(avg_precip)

print("\nTop-10% threshold:")
print(threshold)

print("\nIs this a top-10% wettest month?")
print(avg_precip >= threshold)

National mean precipitation in 2031-12:
221.00581916825254

Top-10% threshold:
334.18516374257155

Is this a top-10% wettest month?
False


In [23]:
# ── Does the resource file predict higher disruption in wettest months? ────
_df_pred = pd.read_csv(
    resourcefilepath / "climate_change_impacts"
    / "ResourceFile_Precipitation_Disruptions_ssp245_mean_monthly_prediction_weather_by_facility.csv",
    index_col=0,
)

# use the lowest disruption file (facility-level predicted disruption)
_df_dis = pd.read_csv(
    resourcefilepath / "climate_change_impacts"
    / "ResourceFile_Precipitation_Disruptions_ssp245_lowest.csv",
)
_df_dis = _df_dis[_df_dis["RealFacility_ID"].isin(tlo_facilities)]
_df_dis["ym"] = _df_dis["year"].astype(str) + "-" + _df_dis["month"].astype(str).str.zfill(2)
_df_dis["key"] = _df_dis["ym"] + ":" + _df_dis["RealFacility_ID"]
_df_dis["in_high_precip"] = _df_dis["key"].isin(high_precip_fac_months)

print("\nPredicted disruption probability (resource file):")
print(f"  High-precip months:  {_df_dis[_df_dis['in_high_precip']]['mean_all_service'].mean():.4f}")
print(f"  Normal months:       {_df_dis[~_df_dis['in_high_precip']]['mean_all_service'].mean():.4f}")

IndentationError: unexpected indent (532812022.py, line 2)